# 🏆 Part 2: Multi-Model Time-Series Forecasting & Benchmarking
This notebook evaluates and compares:
- **Baseline LSTM** (Technical indicators only)
- **Enhanced LSTM** (Technical indicators + FinBERT sentiment)
- **Support Vector Regressor (SVM)**
- **ARIMA Time Series**
- **Naive Persistence Benchmark**

In [ ]:
%load_ext autoreload
%autoreload 2

from src.config import PipelineConfig, TargetMode
from src.data.fetcher import load_sample_data
from src.data.processor import calculate_technical_indicators, create_enhanced_dataset, prepare_time_series_sequences
from src.sentiment.finbert import aggregate_daily_sentiment
from src.models.lstm import LSTMTrainer
from src.models.baselines import SVMPredictor, ARIMAPredictor, NaivePredictor
from src.models.evaluation import calculate_metrics, compare_models
from src.utils.plotting import plot_model_comparison
import matplotlib.pyplot as plt

### 1. Load Data & Prepare Sequences

In [ ]:
cfg = PipelineConfig(symbol="NVDA", target_mode=TargetMode.PRICE)
stock_df, news_df = load_sample_data(symbol="NVDA")
tech_df = calculate_technical_indicators(stock_df)
daily_sent = aggregate_daily_sentiment(news_df)
enh_df = create_enhanced_dataset(tech_df, daily_sent)

target_col = cfg.get_target_column()
baseline_feats = cfg.get_features(enhanced=False)
enhanced_feats = cfg.get_features(enhanced=True)

X_tr_b, X_te_b, y_tr_b, y_te_b, scaler_b, test_dates = prepare_time_series_sequences(
    tech_df, baseline_feats, target_col, sequence_length=45, test_size=0.2
)
X_tr_e, X_te_e, y_tr_e, y_te_e, scaler_e, _ = prepare_time_series_sequences(
    enh_df, enhanced_feats, target_col, sequence_length=45, test_size=0.2
)

y_test_actual = scaler_b.inverse_transform(y_te_b.reshape(-1, 1)).flatten()
print(f"X_train shape: {X_tr_b.shape}, X_test shape: {X_te_b.shape}")

### 2. Train Models

In [ ]:
predictions = {}

# 1. Baseline LSTM
print("Training Baseline LSTM...")
lstm_b = LSTMTrainer(input_dim=len(baseline_feats), hidden_dim=48, num_layers=2)
lstm_b.fit(X_tr_b, y_tr_b, epochs=20, batch_size=32, verbose=False)
predictions["Baseline LSTM"] = scaler_b.inverse_transform(lstm_b.predict(X_te_b).reshape(-1, 1)).flatten()

# 2. Enhanced LSTM
print("Training Sentiment-Enhanced LSTM...")
lstm_e = LSTMTrainer(input_dim=len(enhanced_feats), hidden_dim=48, num_layers=2)
lstm_e.fit(X_tr_e, y_tr_e, epochs=20, batch_size=32, verbose=False)
predictions["Enhanced LSTM (FinBERT)"] = scaler_e.inverse_transform(lstm_e.predict(X_te_e).reshape(-1, 1)).flatten()

# 3. SVM
svm = SVMPredictor(C=50.0)
svm.fit(X_tr_b, y_tr_b)
predictions["Support Vector Regressor (SVM)"] = scaler_b.inverse_transform(svm.predict(X_te_b).reshape(-1, 1)).flatten()

# 4. ARIMA
arima = ARIMAPredictor(order=(3, 1, 0))
arima.fit(y_tr_b)
predictions["ARIMA"] = scaler_b.inverse_transform(arima.forecast(steps=len(y_te_b), last_history=y_tr_b).reshape(-1, 1)).flatten()

# 5. Naive Benchmark
predictions["Naive Persistence"] = NaivePredictor.predict(y_test_actual)
print("Training completed.")

### 3. Evaluate & Compare Leaderboard

In [ ]:
leaderboard = compare_models(predictions, y_test_actual)
leaderboard

### 4. Forecast Visualizations

In [ ]:
fig = plot_model_comparison(
    dates=test_dates,
    actual=y_test_actual,
    predictions=predictions,
    symbol="NVDA",
    target_label="Price ($)"
)
plt.show()